In [2]:
import numpy as np
# import sys
# sys.path.append('../modules/') 
# from config import hsd, order, num_threshold

hsd = 2
order = 1
num_threshold = 1e-10

'''
Create the MPO of a given rule
'''

R = np.arange(1,17).reshape((4,4))


# R = rule_matrix(rule)
bonds = []
tensors = []

r = np.reshape(
        np.transpose(
            np.reshape(
                R, tuple(2*(order+1)*[hsd])
                ),
            tuple(np.array([[i,i+order+1] for i in range(order+1)]).ravel())
            ),
        (hsd**2,hsd**(2*order))
    )
print(tuple(2*(order+1)*[hsd]))
print(tuple(np.array([[i,i+order+1] for i in range(order+1)]).ravel()))
print((hsd**2,hsd**(2*order)))

u, s, v = np.linalg.svd(r, full_matrices=False) 
d = np.sum(s>num_threshold)
bonds.append(d)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]
mpo = np.transpose(np.reshape(U, (hsd,hsd,d)),(0,1,2)) 
tensors.append(mpo)

# for i in range(order-1):
#     r = np.reshape(np.tensordot(S,V,axes=[1,0]),((hsd**2)*d,int(V.shape[-1]/(hsd**2))))
#     u, s, v = np.linalg.svd(r, full_matrices=False) 
#     d = np.sum(s>num_threshold)
#     bonds.append(d)
#     U = u[:,:d]
#     S = np.diag(s[:d])
#     V = v[:d,:]
#     mpo = np.transpose(np.reshape(U, (bonds[i],hsd,hsd,d)),(2,1,0,3)) 
#     tensors.append(mpo)

mpo = np.transpose(np.reshape(np.tensordot(S,V,axes=[1,0]), (d,hsd,hsd)),(2,1,0)) 
tensors.append(mpo)


print(R)
print(r)
print(tensors)

(2, 2, 2, 2)
(0, 2, 1, 3)
(4, 4)
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]
[[ 1  2  5  6]
 [ 3  4  7  8]
 [ 9 10 13 14]
 [11 12 15 16]]
[array([[[-0.19907483, -0.76027381],
        [-0.30094698, -0.54017021]],

       [[-0.60656342,  0.12014059],
        [-0.70843557,  0.34024419]]]), array([[[-14.35377774,   2.44316692],
        [-21.61386089,  -0.91707007]],

       [[-16.16879853,   1.60310767],
        [-23.42888168,  -1.75712932]]])]


In [8]:
# ongedaan maken doe je als:
# .transpose((2,3,1,0)).transpose((3,2,0,1))
# .transpose((3,2,1,0)).transpose((3,2,1,0))

R.reshape((2,2,2,2)).transpose((1,0,3,2))[1][1][1][0]
# [i][j][k][l], where |ij> and |kl>, with flipped binary counting |01> = state 2
# i ==#== j

# door de reshape wordt hier van gemaakt:
# j --#-- l
# i --#-- k

# daar maken we van:
# i --##-- k
# j --##-- l

s1 = R.reshape((2,2,2,2)).transpose((1,0,3,2))

# NEXT
# dan moeten we nu naar 
# i --##-- j
# k --##-- l

s2 = s1.transpose((0,2,1,3)).reshape((4,4))
print(s2)
# NEXT
# svd deze matrix 

u, s, v = np.linalg.svd(s2, full_matrices=False)
d = np.sum(s>num_threshold)
bonds.append(d)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]

# NEXT
# breng in MPO vorm
t1 = U.reshape(2,2,d).transpose((1,0,2)) # gecheckt
t2 = np.tensordot(S,V,axes=[1,0]).reshape((d,2,2)).transpose(2,1,0) # gecheckt

# t1:    t2:
# 
# i        i
# |        |
# #--k  k--#
# |        |
# j        j

# NEXT
# contract ze weer tot een matrix
s3 = np.tensordot(t1,t2,axes=[2,2]) \
                .transpose((0,2,1,3)) \
                                .transpose((2,3,0,1)) \
                                             .transpose((1,0,3,2)).reshape((4,4))
# i    k        i j               k l          l k                  j
# |    |        ###               ###          ###                  #
# #----#  ->    k l        ->     i j    ->    j i           ->     i
# |    |
# j    l
print(s3)
print(np.tensordot(t1,t2,axes=[2,2]).transpose((3,1,2,0)).reshape((4,4)))

[[ 1  3  9 11]
 [ 2  4 10 12]
 [ 5  7 13 15]
 [ 6  8 14 16]]
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]


In [63]:
######
## de correcte manier om het te doen voor (hsd,order) = (2,1)
########

q1 = R.reshape((2,2,2,2),order='F').transpose((0,2,1,3)).reshape((4,4),order='F')
# i j      i k    i-#-j
# ###  ->  ### ==   #    -> i-#-j
# k l      j l    k-#-l


u, s, v = np.linalg.svd(q1, full_matrices=False)
d = np.sum(s>num_threshold)
bonds.append(d)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]

t1 = U.reshape(2,2,d,order='F')
t2 = np.tensordot(S,V,axes=[1,0]).reshape((d,2,2),order='F').transpose(1,2,0) 

s3 = np.tensordot(t1,t2,axes=[2,2]).transpose((0,2,1,3)).reshape((4,4),order='F')
print(s3) 

# # i      i        i k         i j
# # #-k  k-#   ->   ###   ->    ###
# # j      j        j l         k l



[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]


In [213]:
# (hsd,order) = (2,2)
R = np.arange(1,8*8 +1).reshape((8,8))

q1 = R.reshape((2,2,2,2,2,2),order='F')
# i j k  0 1 2 
# ##### 
# l m n  3 4 5


q1 = q1.transpose((0,3,1,2,4,5))
#
# i-#-j        0  2
#   #-k           3
#   #-m           4
# l-#-n        1  5
#

q1 = q1.reshape((2*2,2*2*2*2),order='F')

u, s, v = np.linalg.svd(q1, full_matrices=False)
d = np.sum(s>num_threshold)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]

t1 = U.reshape(2,2,d,order='F')

q2 = np.tensordot(S,V,axes=[1,0])
q2 = q2.reshape(d,2,2,2,2,order='F')

#
#

q2 = q2.transpose((0,1,3,2,4))
q2 = q2.reshape(d*2*2,2*2,order='F')

# #   #-j              d-#-k
# # d-#-k          ->  j-#-n
# #   #-m              m-#
# #   #-n

u, s, v = np.linalg.svd(q2, full_matrices=False)
d2 = np.sum(s>num_threshold)
U = u[:,:d2]
S = np.diag(s[:d2])
V = v[:d2,:]

t2 = U.reshape(d,2,2,d2,order='F').transpose((1,2,0,3))
#  d-#      0              d2      3
#  j-#-d2   1   3   ->   j-#-m   0   1
#  m-#      2              d       2


t3 = np.tensordot(S,V,axes=[1,0]).reshape((d2,2,2),order='F').transpose(1,2,0) 
#     #-k       1? 
#  d2-#       0  
#     #-n       2?


print(t1.shape)
print(t2.shape)
print(t3.shape)

q3 = np.tensordot(t1,t2,axes=[2,2])
q3 = np.tensordot(q3,t3,axes=[4,2])
q3 = q3.transpose((0,2,4,1,3,5))
q3 = q3.reshape((8,8),order='F')

print(q3)


(2, 2, 2)
(2, 2, 2, 2)
(2, 2, 2)
[[ 1.  2.  3.  4.  5.  6.  7.  8.]
 [ 9. 10. 11. 12. 13. 14. 15. 16.]
 [17. 18. 19. 20. 21. 22. 23. 24.]
 [25. 26. 27. 28. 29. 30. 31. 32.]
 [33. 34. 35. 36. 37. 38. 39. 40.]
 [41. 42. 43. 44. 45. 46. 47. 48.]
 [49. 50. 51. 52. 53. 54. 55. 56.]
 [57. 58. 59. 60. 61. 62. 63. 64.]]


In [215]:
print(
    np.tensordot(np.tensordot(
        t1[0][0][:],
        t2[0][0][:][:],axes=[0,0]),
        t3[0][1][:]
        ,axes=[0,0])
)

5.000000000000002


In [216]:
######
## de correcte manier om het te doen voor (hsd,order) = (3,1)
########

R = np.arange(1,9*9 +1).reshape((9,9))

q1 = R.reshape((3,3,3,3),order='F').transpose((0,2,1,3)).reshape((9,9),order='F')
# i j      i k    i-#-j
# ###  ->  ### ==   #    -> i-#-j
# k l      j l    k-#-l


u, s, v = np.linalg.svd(q1, full_matrices=False)
d = np.sum(s>num_threshold)
bonds.append(d)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]

t1 = U.reshape(3,3,d,order='F')
t2 = np.tensordot(S,V,axes=[1,0]).reshape((d,3,3),order='F').transpose(1,2,0) 

s3 = np.tensordot(t1,t2,axes=[2,2]).transpose((0,2,1,3)).reshape((9,9),order='F')
print(s3) 

# # i      i        i k         i j
# # #-k  k-#   ->   ###   ->    ###
# # j      j        j l         k l


[[ 1.  2.  3.  4.  5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14. 15. 16. 17. 18.]
 [19. 20. 21. 22. 23. 24. 25. 26. 27.]
 [28. 29. 30. 31. 32. 33. 34. 35. 36.]
 [37. 38. 39. 40. 41. 42. 43. 44. 45.]
 [46. 47. 48. 49. 50. 51. 52. 53. 54.]
 [55. 56. 57. 58. 59. 60. 61. 62. 63.]
 [64. 65. 66. 67. 68. 69. 70. 71. 72.]
 [73. 74. 75. 76. 77. 78. 79. 80. 81.]]


In [217]:
# (hsd,order) = (3,2)
R = np.arange(1,27*27 +1).reshape((27,27))

q1 = R.reshape((3,3,3,3,3,3),order='F')
# i j k  0 1 2 
# ##### 
# l m n  3 4 5


q1 = q1.transpose((0,3,1,2,4,5))
#
# i-#-j        0  2
#   #-k           3
#   #-m           4
# l-#-n        1  5
#

q1 = q1.reshape((3*3,3*3*3*3),order='F')

u, s, v = np.linalg.svd(q1, full_matrices=False)
d = np.sum(s>num_threshold)
U = u[:,:d]
S = np.diag(s[:d])
V = v[:d,:]

t1 = U.reshape(3,3,d,order='F')

q2 = np.tensordot(S,V,axes=[1,0])
q2 = q2.reshape(d,3,3,3,3,order='F')

#
#

q2 = q2.transpose((0,1,3,2,4))
q2 = q2.reshape(d*3*3,3*3,order='F')

# #   #-j              d-#-k
# # d-#-k          ->  j-#-n
# #   #-m              m-#
# #   #-n

u, s, v = np.linalg.svd(q2, full_matrices=False)
d2 = np.sum(s>num_threshold)
U = u[:,:d2]
S = np.diag(s[:d2])
V = v[:d2,:]

t2 = U.reshape(d,3,3,d2,order='F').transpose((1,2,0,3))
#  d-#      0              d2      3
#  j-#-d2   1   3   ->   j-#-m   0   1
#  m-#      2              d       2


t3 = np.tensordot(S,V,axes=[1,0]).reshape((d2,3,3),order='F').transpose(1,2,0) 
#     #-k       1? 
#  d2-#       0  
#     #-n       2?


print(t1.shape)
print(t2.shape)
print(t3.shape)

q3 = np.tensordot(t1,t2,axes=[2,2])
q3 = np.tensordot(q3,t3,axes=[4,2])
q3 = q3.transpose((0,2,4,1,3,5))
q3 = q3.reshape((27,27),order='F')

print(q3)


(3, 3, 2)
(3, 3, 2, 2)
(3, 3, 2)
[[  1.   2.   3.   4.   5.   6.   7.   8.   9.  10.  11.  12.  13.  14.
   15.  16.  17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.]
 [ 28.  29.  30.  31.  32.  33.  34.  35.  36.  37.  38.  39.  40.  41.
   42.  43.  44.  45.  46.  47.  48.  49.  50.  51.  52.  53.  54.]
 [ 55.  56.  57.  58.  59.  60.  61.  62.  63.  64.  65.  66.  67.  68.
   69.  70.  71.  72.  73.  74.  75.  76.  77.  78.  79.  80.  81.]
 [ 82.  83.  84.  85.  86.  87.  88.  89.  90.  91.  92.  93.  94.  95.
   96.  97.  98.  99. 100. 101. 102. 103. 104. 105. 106. 107. 108.]
 [109. 110. 111. 112. 113. 114. 115. 116. 117. 118. 119. 120. 121. 122.
  123. 124. 125. 126. 127. 128. 129. 130. 131. 132. 133. 134. 135.]
 [136. 137. 138. 139. 140. 141. 142. 143. 144. 145. 146. 147. 148. 149.
  150. 151. 152. 153. 154. 155. 156. 157. 158. 159. 160. 161. 162.]
 [163. 164. 165. 166. 167. 168. 169. 170. 171. 172. 173. 174. 175. 176.
  177. 178. 179. 180. 181. 182. 183. 184. 185. 186. 187

In [262]:
# (hsd,order) = (2,3)

hsd = 2
order = 3
d = hsd**(order+1)


R = np.arange(1,d*d +1).reshape((d,d))

q1 = R.reshape(tuple(2*(order+1)*[hsd]),order='F')
# i j k  0 1 2 
# ##### 
# l m n  3 4 5

tp1 = [0]+[(order+1)]+list(range(1,order+1))+list(range(order+2,2*(order+1)))
q1 = q1.transpose(tp1)

# #
# # i-#-j        0  2
# #   #-k           3
# #   #-m           4
# # l-#-n        1  5
# #

q1 = q1.reshape((hsd*hsd,-1),order='F')

u, s, v = np.linalg.svd(q1, full_matrices=False)
b1 = np.sum(s>num_threshold)
U = u[:,:b1]
S = np.diag(s[:b1])
V = v[:b1,:]

t1 = U.reshape(hsd,hsd,b1,order='F')

q2 = np.tensordot(S,V,axes=[1,0])
print(q2.shape)
q2 = q2.reshape(tuple([b1]+ (2*(order+1)-2)*[hsd]),order='F')
print(q2.shape)
tp2 = [0,1,order+1]+list(range(2,order+1))+list(range(order+2,2*(order+1)-1))
print(tp2)
q2 = q2.transpose(tp2)
q2 = q2.reshape((b1*hsd*hsd,-1),order='F')
print(q2.shape)

# #   #-j              d-#-k
# # d-#-k          ->  j-#-n
# #   #-m              m-#
# #   #-n

u, s, v = np.linalg.svd(q2, full_matrices=False)
b2 = np.sum(s>num_threshold)
U = u[:,:b2]
S = np.diag(s[:b2])
V = v[:b2,:]

t2 = U.reshape(b1,hsd,hsd,b2,order='F').transpose((1,2,0,3))
#  d-#      0              d2      3
#  j-#-d2   1   3   ->   j-#-m   0   1
#  m-#      2              d       2

q3 = np.tensordot(S,V,axes=[1,0])
print(q3.shape)

# insert code from (hsd,order) = (2,2)

q3 = q3.reshape((b2,hsd,hsd,hsd,hsd),order='F')

q3 = q3.transpose((0,1,3,2,4))
q3 = q3.reshape((b2*hsd*hsd,-1),order='F')


u, s, v = np.linalg.svd(q3, full_matrices=False)
b3 = np.sum(s>num_threshold)
U = u[:,:b3]
S = np.diag(s[:b3])
V = v[:b3,:]

t3 = U.reshape(b2,hsd,hsd,b3,order='F').transpose((1,2,0,3))
#  d-#      0              d2      3
#  j-#-d2   1   3   ->   j-#-m   0   1
#  m-#      2              d       2


t4 = np.tensordot(S,V,axes=[1,0]).reshape((b3,hsd,hsd),order='F').transpose(1,2,0) 
#     #-k       1? 
#  d2-#       0  
#     #-n       2?

print(t1.shape)
print(t2.shape)
print(t3.shape)
print(t4.shape)

c1 = np.tensordot(t1,t2,axes=[2,2])
c2 = np.tensordot(c1,t3,axes=[4,2])
c3 = np.tensordot(c2,t4,axes=[6,2])
c3 = c3.transpose((0,2,4,6,1,3,5,7))
c3 = c3.reshape((d,d),order='F')

print(c3)

(2, 64)
(2, 2, 2, 2, 2, 2, 2)
[0, 1, 4, 2, 3, 5, 6]
(8, 16)
(2, 16)
(8, 4)
(2, 2, 2)
(2, 2, 2, 2)
(2, 2, 2, 2)
(2, 2, 2)
[[  1.   2.   3.   4.   5.   6.   7.   8.   9.  10.  11.  12.  13.  14.
   15.  16.]
 [ 17.  18.  19.  20.  21.  22.  23.  24.  25.  26.  27.  28.  29.  30.
   31.  32.]
 [ 33.  34.  35.  36.  37.  38.  39.  40.  41.  42.  43.  44.  45.  46.
   47.  48.]
 [ 49.  50.  51.  52.  53.  54.  55.  56.  57.  58.  59.  60.  61.  62.
   63.  64.]
 [ 65.  66.  67.  68.  69.  70.  71.  72.  73.  74.  75.  76.  77.  78.
   79.  80.]
 [ 81.  82.  83.  84.  85.  86.  87.  88.  89.  90.  91.  92.  93.  94.
   95.  96.]
 [ 97.  98.  99. 100. 101. 102. 103. 104. 105. 106. 107. 108. 109. 110.
  111. 112.]
 [113. 114. 115. 116. 117. 118. 119. 120. 121. 122. 123. 124. 125. 126.
  127. 128.]
 [129. 130. 131. 132. 133. 134. 135. 136. 137. 138. 139. 140. 141. 142.
  143. 144.]
 [145. 146. 147. 148. 149. 150. 151. 152. 153. 154. 155. 156. 157. 158.
  159. 160.]
 [161. 162. 163. 164. 165. 16

In [396]:
# now generalize it for all (hsd,order)

hsd = 2
order = 1

# CHECK
#
#     order  1  2  3  4 | 
# hsd                   |
#  2         x  x  4  x |
#  3         x  x  4  x |
#  4         x  x  4  x |

d = hsd**(order+1)
R = np.arange(1,d*d+1).reshape((d,d))

def operator_to_mpo(operator):
    d           = operator.shape[0]
    tensors     = []
    bonds       = []
    order       = int(np.rint(np.log(d)/np.log(hsd)-1))

    ## INITIALIZE
    T = operator.reshape(tuple(2*(order+1)*[hsd]),order='F')
    # i j k  0 1 2 
    # ##### 
    # l m n  3 4 5

    tp = [0]+[(order+1)]+list(range(1,order+1))+list(range(order+2,2*(order+1)))
    T = T.transpose(tp).reshape((hsd*hsd,-1),order='F')
    # # i-#-j        0  2
    # #   #-k           3
    # #   #-m           4
    # # l-#-n        1  5




    ## FIRST DECOMPOSITION
    u, s, v = np.linalg.svd(T, full_matrices=False)
    bonds.append(np.sum(s>num_threshold))
    U = u[:,:bonds[-1]]
    S = np.diag(s[:bonds[-1]])
    V = v[:bonds[-1],:]

    tensors.append(U.reshape(hsd,hsd,bonds[-1],order='F'))

    if order ==1:
        tensors.append(np.tensordot(S,V,axes=[1,0]).reshape((bonds[-1],hsd,hsd),order='F').transpose(1,2,0))
        return tensors 
    
    else:
        tp = tuple([bonds[-1]]+ (2*(order+1)-2)*[hsd])
        T = np.tensordot(S,V,axes=[1,0]).reshape(tp,order='F')
        tp = [0,1,order+1]+list(range(2,order+1))+list(range(order+2,2*(order+1)-1))
        T = T.transpose(tp).reshape((bonds[-1]*hsd*hsd,-1),order='F')
        # #   #-j              d-#-k
        # # d-#-k          ->  j-#-n
        # #   #-m              m-#
        # #   #-n
        


        for x in range(order-2):
            ## SECOND DECOMPOSITION
            u, s, v = np.linalg.svd(T, full_matrices=False)
            bonds.append(np.sum(s>num_threshold))
            U = u[:,:bonds[-1]]
            S = np.diag(s[:bonds[-1]])
            V = v[:bonds[-1],:]

            tensors.append(U.reshape(bonds[-2],hsd,hsd,bonds[-1],order='F').transpose((1,2,0,3)))
            #  d-#      0              d2      3
            #  j-#-d2   1   3   ->   j-#-m   0   1
            #  m-#      2              d       2

            
            tp = tuple([bonds[-1]]+[hsd]*(2*(order-1-x)))
            T = np.tensordot(S,V,axes=[1,0]).reshape(tp,order='F')
            tp = tuple([0,1,order-x]+list(range(2,order-x))+list(range(order+1-x,2*(order-x)-1)))
            T = T.transpose(tp).reshape((bonds[-1]*hsd*hsd,-1),order='F')






        ## FINAL DECOMPOSITION
        u, s, v = np.linalg.svd(T, full_matrices=False)
        bonds.append(np.sum(s>num_threshold))
        U = u[:,:bonds[-1]]
        S = np.diag(s[:bonds[-1]])
        V = v[:bonds[-1],:]

        tensors.append(U.reshape(bonds[-2],hsd,hsd,bonds[-1],order='F').transpose((1,2,0,3)))
        #  d-#      0              d2      3
        #  j-#-d2   1   3   ->   j-#-m   0   1
        #  m-#      2              d       2

        tensors.append(np.tensordot(S,V,axes=[1,0]).reshape((bonds[-1],hsd,hsd),order='F').transpose(1,2,0))
        #     #-k       1
        #  d2-#       0  
        #     #-n       2

        return tensors

def mpo_to_operator(tensors):
    t1 = tensors[0]
    i  = 2
    for t2 in tensors[1:]:
        t1 =  np.tensordot(t1,t2,axes=[i,2])
        i  += 2
    return t1.transpose(list(range(0,2*(order+1),2))+list(range(1,2*(order+1),2))).reshape((d,d),order='F')



tensors = operator_to_mpo(R)
operator = mpo_to_operator(tensors)


print(np.allclose(operator,R))

True
